# [3장 2강] 실습: 분류 평가지표

## 실습 목표

- 고객 이탈 여부를 예측하는 이진 분류 문제를 정의할 수 있다.
- 수치형·범주형 Feature를 분류 모델에 맞게 전처리할 수 있다.
- Logistic Regression으로 고객 이탈 여부와 확률을 예측할 수 있다.
- Confusion Matrix의 TN, FP, FN, TP를 구분할 수 있다.
- Accuracy, Precision, Recall, F1-score와 AUC의 차이를 설명할 수 있다.
- 불균형 데이터에서 Accuracy만으로 평가하면 안 되는 이유를 설명할 수 있다.
- Threshold 변화에 따른 Precision–Recall Trade-off를 해석할 수 있다.

## 사용 데이터

- 파일: `Telco-Customer.csv`
- Label: `Churn` (`No`: 유지, `Yes`: 이탈)
- Positive Class: 이탈 고객 `Yes=1`

## 진행 방식

- 문제 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- 결과값을 근거로 문제에 제시된 질문에 답합니다.

## 실습 준비: 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [ ]:
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path='/content/drive/MyDrive/Telco-Customer.csv'
except ModuleNotFoundError:
    file_path='Telco-Customer.csv'

telco_df=pd.read_csv(file_path)
display(telco_df.head())
print('데이터 크기:',telco_df.shape)
telco_df.info()

## 필수 1: 고객 이탈 분류를 위한 데이터 준비

### 문제 1-1: Label 변환과 전처리 Pipeline 구성하기

#### 문제 설명

통신사는 고객 정보와 서비스 이용 정보를 이용해 이탈 여부를 예측하려고 합니다. 식별자를 제거하고, 숫자로 저장되지 않은 `TotalCharges`를 변환한 뒤 수치형·범주형 Feature를 모델이 학습할 수 있도록 전처리합니다.

#### 요구사항

1. `Churn`의 클래스별 개수와 비율을 출력합니다.
2. `customerID`를 입력 Feature에서 제거합니다.
3. `TotalCharges`를 숫자로 변환하고 변환 후 결측치 수를 출력합니다.
4. `Churn`에서 `Yes=1`, `No=0`으로 변환합니다.
5. X와 y를 분리하고 `random_state=42`, `stratify=y`로 학습 80%, 테스트 20%로 나눕니다.
6. 수치형은 중앙값 대체 후 MinMaxScaler, 범주형은 OneHotEncoder를 적용하는 ColumnTransformer를 만듭니다.
7. 전처리기를 학습 데이터에 `fit_transform()`, 테스트 데이터에 `transform()`합니다.
8. 데이터 크기, Label 분포, 변환 후 Feature 개수와 수치형 값의 범위를 출력합니다.

#### 출력 결과

- 유지 5,174명, 이탈 1,869명
- `TotalCharges` 숫자 변환 후 결측치 11개
- 학습 데이터 5,634행, 테스트 데이터 1,409행
- 전처리 후 Feature 30개

#### 결과 해석 작성

> `stratify=y`를 사용한 이유와 결측치 대체·Scaling·Encoding을 학습 데이터에서 먼저 학습해야 하는 이유는 무엇인가요?

## 필수 2: Logistic Regression 학습 및 평가

### 문제 2-1: Confusion Matrix와 분류 평가지표 확인하기

#### 문제 설명

Logistic Regression으로 고객 이탈 확률과 클래스를 예측합니다. 실제 이탈 고객을 유지 고객으로 잘못 판단하는 FN을 포함해 여러 지표를 함께 확인합니다.

#### 요구사항

1. `LogisticRegression(max_iter=1000)`을 학습합니다.
2. 테스트 데이터의 클래스와 이탈 확률을 예측합니다.
3. Confusion Matrix와 TN, FP, FN, TP를 출력합니다.
4. Accuracy, Precision, Recall, F1-score와 AUC를 계산합니다.
5. 실제값, 예측값과 이탈 확률 앞 10개를 비교합니다.
6. 고객 이탈 문제에서 Accuracy만으로 모델을 평가하면 안 되는 이유를 설명합니다.
7. FN의 의미와 이탈 방지 목적에서 Recall이 중요한 이유를 설명합니다.

#### 결과 해석 작성

> 이 고객 이탈 데이터에서 Accuracy만으로 모델을 평가하면 안 되는 이유는 무엇이며, FN은 어떤 고객을 의미하나요?

## 필수 3: 평가지표 직접 계산 및 기준 모델 비교

### 문제 3-1: Confusion Matrix 공식과 다수 클래스 기준 모델 확인하기

#### 문제 설명

혼동행렬의 네 값으로 Accuracy, Precision, Recall과 F1-score를 직접 계산합니다. 또한 모든 고객을 다수 클래스인 유지 고객으로 예측하는 기준 모델을 만들어 Accuracy가 높더라도 이탈 고객을 찾지 못할 수 있음을 확인합니다.

#### 요구사항

1. TN, FP, FN, TP로 네 가지 지표를 직접 계산합니다.
2. scikit-learn 결과와 일치하는지 확인합니다.
3. 모든 테스트 고객을 `0`으로 예측하는 기준 모델을 만듭니다.
4. 기준 모델의 Confusion Matrix, Accuracy, Precision, Recall과 F1-score를 출력합니다.
5. 기준 모델의 Accuracy가 70%를 넘는데도 유용한 이탈 예측 모델이 아닌 이유를 설명합니다.

#### 결과 해석 작성

> 모든 고객을 유지 고객으로 예측한 기준 모델의 Accuracy가 약 73.5%인데도 이탈 예측 모델로 사용할 수 없는 이유는 무엇인가요?

## 심화 1: 분류 Threshold 변경과 Recall 비교

### 문제 4-1: Threshold 0.5와 0.3 비교하기

#### 문제 설명

이탈 고객을 놓치는 FN을 줄이기 위해 기본 Threshold 0.5와 더 낮은 0.3을 비교합니다. Threshold를 낮추면 더 많은 고객을 이탈로 판단하므로 Recall과 FP가 함께 변할 수 있습니다.

#### 요구사항

1. 이탈 확률에 Threshold 0.5와 0.3을 각각 적용합니다.
2. 두 Confusion Matrix와 TN, FP, FN, TP를 출력합니다.
3. Accuracy, Precision, Recall과 F1-score 비교표를 만듭니다.
4. Threshold를 낮췄을 때 FP와 FN의 변화를 설명합니다.
5. 이탈 방지 캠페인에서 두 Threshold 중 무엇을 선택할지 비용 관점에서 작성합니다.

#### 결과 해석 작성

> Threshold를 0.5에서 0.3으로 낮추면 FP, FN, Precision과 Recall은 어떻게 변하며, 어떤 상황에서 0.3이 더 적절한가요?

## 실습 마무리

1. Accuracy, Precision, Recall과 F1-score는 각각 어떤 질문에 답하나요?
2. AUC는 무엇을 평가하나요?
3. 고객 이탈 문제에서 Recall과 Precision 중 하나만 항상 우선할 수 없는 이유는 무엇인가요?